In [ ]:
"""
Aufgabe 5: Anwendung auf komplexe Datensaetze - Handschrifterkennung (MNIST)
=============================================================================
Wendet den in Aufgabe 4 erlernten Keras-Workflow auf den MNIST-Datensatz an.
Umschalten auf fashion-MNIST: einfach die Import-Zeile unten aendern.
"""

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.datasets import mnist


# ---------------------------------------------------------------------------
# Aufgabe 5.a
# ---------------------------------------------------------------------------
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(f"Trainingsdaten:  {x_train.shape}, Labels: {y_train.shape}")
print(f"Testdaten:       {x_test.shape}, Labels: {y_test.shape}")
print(f"Pixelwerte liegen zwischen {x_train.min()} und {x_train.max()}")
print(f"Vorkommende Klassen: {np.unique(y_train)}")

# Normierung: Pixelwerte liegen als uint8 zwischen 0 und 255.
# Wir skalieren sie auf [0, 1], da grosse absolute Werte in Kombination mit
# den zufaellig initialisierten Gewichten sonst zu instabilem/langsamem
# Training fuehren koennen (vgl. Kapitel 2.2 der Angabe).'
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Die Labels (0-9) bleiben als einfache Integer erhalten. Wir nutzen dafuer
# spaeter "sparse_categorical_crossentropy", die direkt mit Integer-Labels
# arbeitet (keine manuelle One-Hot-Kodierung noetig).

# Ein paar Beispielbilder zur Kontrolle anzeigen
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")
plt.tight_layout()
plt.savefig("aufgabe5_beispielbilder.png", dpi=150)
plt.show()


# ---------------------------------------------------------------------------
# Aufgabe 5.b
# ---------------------------------------------------------------------------
# Output: 10 Neuronen (eines pro Ziffer 0-9).
# Aktivierung "softmax", da wir eine Mehrklassen-Klassifikation (nicht binaer
# wie in Aufgabe 4) haben: softmax liefert eine Wahrscheinlichkeitsverteilung
# ueber alle 10 Klassen, die sich zu 1 aufsummiert.
model_simple = Sequential()
model_simple.add(Input(shape=(28, 28)))
model_simple.add(Flatten())
model_simple.add(Dense(10, activation="softmax"))  # Output-Layer

model_simple.summary()


# ---------------------------------------------------------------------------
# Aufgbae 5.c
# ---------------------------------------------------------------------------
# sparse_categorical_crossentropy: passende Loss-Funktion fuer
# Mehrklassen-Klassifikation mit Integer-Labels (0-9) statt One-Hot-Vektoren.
model_simple.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

history_simple = model_simple.fit(
    x_train, y_train,
    epochs=10,
    validation_split=0.1,
    verbose=1,
)

loss_simple, acc_simple = model_simple.evaluate(x_test, y_test, verbose=0)
print(f"\n[Einfaches Modell] Test-Loss: {loss_simple:.4f}, Test-Genauigkeit: {acc_simple:.2%}")


# ---------------------------------------------------------------------------
# Aufgabe 5d
# ---------------------------------------------------------------------------
y_pred_simple = np.argmax(model_simple.predict(x_test, verbose=0), axis=1)

cm_simple = confusion_matrix(y_test, y_pred_simple)
disp_simple = ConfusionMatrixDisplay(confusion_matrix=cm_simple)
fig, ax = plt.subplots(figsize=(7, 7))
disp_simple.plot(ax=ax, cmap="Blues", colorbar=True)
ax.set_title("Confusion Matrix - Modell ohne Hidden Layer")
plt.tight_layout()
plt.savefig("aufgabe5_confusion_simple.png", dpi=150)
plt.show()


# ---------------------------------------------------------------------------
# Aufgabe 5e
# ---------------------------------------------------------------------------
# Wir testen mehrere Neuronenzahlen im Hidden Layer und vergleichen die
# Test-Genauigkeit.
hidden_layer_sizes = [32, 64, 128]
results = {}

for n_neurons in hidden_layer_sizes:
    model_hidden = Sequential()
    model_hidden.add(Input(shape=(28, 28)))
    model_hidden.add(Flatten())
    # ReLU ist eine ueblich Wahl fuer Hidden Layer: einfach zu berechnen,
    # vermeidet das Vanishing-Gradient-Problem besser als z.B. Sigmoid.
    model_hidden.add(Dense(n_neurons, activation="relu"))
    model_hidden.add(Dense(10, activation="softmax"))  # Output-Layer

    model_hidden.compile(
        loss="sparse_categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"],
    )

    history_hidden = model_hidden.fit(
        x_train, y_train,
        epochs=10,
        validation_split=0.1,
        verbose=0,
    )

    loss_hidden, acc_hidden = model_hidden.evaluate(x_test, y_test, verbose=0)
    results[n_neurons] = (model_hidden, history_hidden, loss_hidden, acc_hidden)
    print(f"[Hidden Layer, {n_neurons:3d} Neuronen] Test-Loss: {loss_hidden:.4f}, "
          f"Test-Genauigkeit: {acc_hidden:.2%}")

# Bestes Modell (hoechste Test-Genauigkeit) fuer die Confusion Matrix waehlen
best_n = max(results, key=lambda n: results[n][3])
best_model, best_history, best_loss, best_acc = results[best_n]
print(f"\nBestes Modell mit Hidden Layer: {best_n} Neuronen, "
      f"Test-Genauigkeit: {best_acc:.2%}")

y_pred_hidden = np.argmax(best_model.predict(x_test, verbose=0), axis=1)
cm_hidden = confusion_matrix(y_test, y_pred_hidden)
disp_hidden = ConfusionMatrixDisplay(confusion_matrix=cm_hidden)
fig, ax = plt.subplots(figsize=(7, 7))
disp_hidden.plot(ax=ax, cmap="Blues", colorbar=True)
ax.set_title(f"Confusion Matrix - Hidden Layer ({best_n} Neuronen)")
plt.tight_layout()
plt.savefig("aufgabe5_confusion_hidden.png", dpi=150)
plt.show()

# Vergleich der Test-Genauigkeit: ohne vs. mit Hidden Layer
print("\n--- Vergleich ---")
print(f"Ohne Hidden Layer:              {acc_simple:.2%}")
for n_neurons, (_, _, _, acc) in results.items():
    print(f"Mit Hidden Layer ({n_neurons:3d} Neuronen): {acc:.2%}")
